In [1]:
# set wd
import os
from pathlib import Path
print(os.getcwd())
wd = Path(os.getcwd())
if wd.name == "notebooks":
    %cd ..
print(f"Working Dir Base: {(os.getcwd())}")

/Users/peli/Projects/Repositories/MEGPypes/notebooks
/Users/peli/Projects/Repositories/MEGPypes
Working Dir Base: /Users/peli/Projects/Repositories/MEGPypes


In [2]:
import yaml
import time
from bids.layout import BIDSLayout
# import
from megpypes.pipelines.meg_preprocessing import create_meg_preprocessing


/Users/peli/Projects/Repositories/MEGPypes/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded a MEG dataset from here:
https://openneuro.org/datasets/ds006629/versions/1.0.1

using datalad
```
datalad install -s https://github.com/OpenNeuroDatasets/ds006629.git data/ds006629
```
```
cd output/ds006629
datalad get -r .
```


First we need to grab data from our dataset.
We assume the data operate within a bids compliant data structure
https://peerherholz.github.io/workshop_weizmann/nipype/notebooks/basic_data_input_bids.html



In [3]:
# do some data inspection using pybids?
layout = BIDSLayout("data/ds006629/")
print(layout)
# subjects
subjects = layout.get_subjects()
print(f"Subjects: {subjects}")
# datatypes
bidstypes = layout.get_datatypes()
print(f"Data types: {bidstypes}")
# suffixes
print(layout.get_suffixes(datatype='func'))
# see tasks
layout.get_tasks()
# see dataset description
layout.get_dataset_description()
#

# see data metadata


BIDS Layout: ...itories/MEGPypes/data/ds006629 | Subjects: 19 | Sessions: 0 | Runs: 19
Subjects: ['01', '02', '04', '05', '06', '07', '08', '09', '10', '11', '12', '14', '15', '16', '17', '18', '19', '20', '21']
Data types: ['meg']
[]


{'Name': 'SINGSING',
 'BIDSVersion': '1.7.0',
 'License': 'CC0',
 'DatasetType': 'raw',
 'Authors': ['Valerie Chanoine',
  'Jean-Michel Badier',
  'Mireille Besson',
  'Talya Inbar'],
 'Acknowledgements': 'MEG data acquisition was performed in the MEG Centre (Timone Hospital, Marseille, France)',
 'Funding': ['This research has been supported by funding from the Institute of Convergence ILCB (France 2030, ANR-16-CONV-0002) and the Excellence Initiative of Aix-Marseille University A*MIDEX (ANR-11-IDEX-0001-02)'],
 'ReferencesAndLinks': ['a data paper',
  'a resource to be cited when using the data'],
 'DatasetDOI': 'doi:10.18112/openneuro.ds006629.v1.0.1',
 'GeneratedBy': [{'Name': 'MNE-BIDS',
   'Version': '0.14',
   'Description': 'MNE-BIDS is a Python package that allows you to read and write BIDS-compatible datasets with the help of MNE-Python.'}],
 'SourceDatasets': [{'DOI': 'doi:10.18112/openneuro.ds006629.v1.0.0',
   'URL': 'https://openneuro.org/datasets/ds006629',
   'Version':

In [4]:
# Load configs
import os
import yaml
from nipype import config as nconfig

config_path = "config/config_ds006629.yaml"
with open(config_path, "r") as yamlfile:
    config = yaml.load(yamlfile, Loader=yaml.FullLoader)

wf_config = config['workflow']
paths_config = config['paths']

# Configure Nipype logging (applies to all subprocesses)
nconfig.update_config({
    'logging': {
        'log_directory': os.path.join(paths_config["workdir"], 'logs'),
        'log_to_file': True,
        'interface_level': 'info',
        'workflow_level': 'info',
    },
    'execution': {
        'crashdump_dir': os.path.abspath('crashes'),
        'remove_unnecessary_outputs': False,
    }
})

# Create workflow
wf = create_meg_preprocessing(
    basedir=paths_config['basedir'], 
    workdir=paths_config['workdir'], 
    output_dir=paths_config['outputdir'], 
    subject_list=paths_config['subjects'],
    pipeline_config=config['pipeline_config']
    )

# visualize workflow graph
wf.write_graph(graph2use='colored', simple_form=True)
print(f"Workflow graph saved to: {wf.base_dir}/megpreproc/graph.png")

# Run workflow
n_workers = wf_config.get("n_workers", max(1, os.cpu_count() - 2))
print(f"Running with {n_workers} workers")

result = wf.run(
    plugin=wf_config["plugin"],
    plugin_args={"n_procs": n_workers}
)

raw_dir /Users/peli/Projects/Repositories/MEGPypes/data/ds006629
Valid inputs for initial_preproc: {'h_freq', 'max_buffer', 'trait_added', 'out_file', 'stim_channel', 'min_buffer', 'gradcomp_order', 'trait_modified', 'gradcomp_auto', 'l_freq', 'in_file'}
Step 'crop' argument 'stim_channel': None
Step 'crop' argument 'min_buffer': -0.2
Step 'crop' argument 'max_buffer': 0.5
Step 'filter' argument 'l_freq': 1.0
Step 'filter' argument 'h_freq': 100
Step 'gradcomp' argument 'gradcomp_auto': True
Step 'gradcomp' argument 'order': 3
Valid inputs for artifact_rejection: {'trait_added', 'n_chunks', 'win_sz', 'mag_only', 'out_file', 'nfft', 'n_iter_max', 'trait_modified', 'fline', 'enable_zapline', 'detect_line_freq', 'in_file', 'spot_sz'}
Step 'zapline' argument 'fline': 60.0
Step 'zapline' argument 'n_chunks': 10
Step 'zapline' argument 'spot_sz': 7
Step 'zapline' argument 'win_sz': 12
Step 'zapline' argument 'nfft': 2048
Step 'zapline' argument 'n_iter_max': 100
Step 'zapline' argument 'mag_

2026-03-07 11:30:57,589 [INFO] megpypes.interfaces.artifact_rejection: NODE: Artifact Rejection | In-File: /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-01/initial_preproc/initial_preproc_raw.fif


Opening raw data file /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-01/initial_preproc/initial_preproc_raw.fif...
    Read a total of 1 projection items:
        axial-Raw-0.000-900.800-PCA-01 (1 x 229)  idle
    Range : 10397 ... 205828 =     41.588 ...   823.312 secs
Ready.
Reading 0 ... 195431  =      0.000 ...   781.724 secs...
Power of components removed by DSS: 0.00
Iteration 0 score: 4.642138213188996e-35
Power of components removed by DSS: 0.00
Iteration 1 score: 4.6325539095640306e-35
Power of components removed by DSS: 0.00
Iteration 2 score: 4.6390204348304375e-35
Power of components removed by DSS: 0.00
Iteration 3 score: 4.640679921704973e-35
Power of components removed by DSS: 0.00
Iteration 4 score: 3.2210728698067197e-35
Power of components removed by DSS: 0.00
Iteration 5 score: 3.119522285199703e-35
Power of components removed by DSS: 0.00
Iteration 6 score: 3.1151912285389766e-35
Power of components removed by DSS: 0.00
Iteration 7 sco

2026-03-07 11:31:55,442 [INFO] megpypes.interfaces.artifact_rejection: NODE: Artifact Rejection | In-File: /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-02/initial_preproc/initial_preproc_raw.fif


Opening raw data file /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_subject_id_sub-02/initial_preproc/initial_preproc_raw.fif...
    Read a total of 1 projection items:
        axial-Raw-0.000-813.092-PCA-01 (1 x 231)  idle
    Range : 1838 ... 196260 =      7.352 ...   785.040 secs
Ready.
Reading 0 ... 194422  =      0.000 ...   777.688 secs...
Power of components removed by DSS: 0.00
Iteration 0 score: 4.2596915124677625e-35
Power of components removed by DSS: 0.00
Iteration 1 score: 4.260615749511549e-35
Power of components removed by DSS: 0.00
Iteration 2 score: 4.264979251775285e-35
Power of components removed by DSS: 0.00
Iteration 3 score: 4.268007200930491e-35
Power of components removed by DSS: 0.00
Iteration 4 score: 4.269347857746125e-35
Power of components removed by DSS: 0.00
Iteration 5 score: 3.593056598981154e-35
Power of components removed by DSS: 0.00
Iteration 6 score: 3.591412472558803e-35
Power of components removed by DSS: 0.00
Iteration 7 score: 

RuntimeError: 2 raised. Re-raising first.